# Image -> image: benchmark production

Objectif: tester la recherche image -> image generale sans remplacer le notebook `01_filter_by_image_insightface_qdrant.ipynb`.

Le notebook `01` reste la reference pour le filtre par personne. Celui-ci sert a visualiser les strategies globales image embedding et image+caption.

## Deux modes produit

```text
reference image
-> mode person filter: InsightFace/Qdrant si un visage est detecte
-> mode visual similarity: image embedding OpenCLIP
-> optional metadata filters
-> optional caption/context boost
-> results
```

Ces modes doivent rester separes car ils ne repondent pas a la meme question produit.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path.cwd()
while ROOT.name != 'photo-ai-platform' and ROOT.parent != ROOT:
    ROOT = ROOT.parent

BENCHMARK = ROOT / 'ml' / 'experiments' / 'artifact_benchmark.py'
REPORT = ROOT / 'reports' / 'algorithm_tests' / 'latest' / 'summary.json'
BENCHMARK.exists(), REPORT


(True,
 WindowsPath('c:/Users/choun/OneDrive - Groupe INSEEC (POCE)/Bureau/projet-ia/photo-ai-platform/reports/algorithm_tests/latest/summary.json'))

In [2]:
cmd = [sys.executable, str(BENCHMARK), '--top-k', '10', '--text-batch-size', '8']
subprocess.run(cmd, cwd=ROOT, check=True)


CompletedProcess(args=['c:\\Users\\choun\\miniconda3\\envs\\env\\python.exe', 'c:\\Users\\choun\\OneDrive - Groupe INSEEC (POCE)\\Bureau\\projet-ia\\photo-ai-platform\\ml\\experiments\\artifact_benchmark.py', '--top-k', '10', '--text-batch-size', '8'], returncode=0)

In [3]:
summary = json.loads(REPORT.read_text(encoding='utf-8'))
summary['image_to_image']['metrics']


{'duplicate_groups': 0,
 'duplicate_hit_at_1': None,
 'duplicate_hit_at_5': None,
 'caption_overlap_precision_at_k_visual': 0.2767,
 'caption_overlap_precision_at_k_hybrid': 0.4567}

In [4]:
queries = summary['image_to_image']['queries']
first = queries[0]
{
    'query': first['query']['caption'],
    'visual_top_5': [item['caption'] for item in first['methods']['image_embedding_only'][:5]],
    'hybrid_top_5': [item['caption'] for item in first['methods']['image_caption_hybrid'][:5]],
}


{'query': 'a living room with a television and a fireplace',
 'visual_top_5': ['a kitchen with a stove, sink, microwave, and dishwasher',
  'a kitchen with a sink and three stools',
  'a kitchen with a stove, sink, and a refrigerator',
  'a living room with a couch, a table, and a chair',
  'a kitchen counter with a sink and a sink'],
 'hybrid_top_5': ['a living room with a couch and a television',
  'a living room with a couch and a television',
  'a living room with a couch, chair, and fireplace',
  'a living room with a couch, a table, and a chair',
  'a living room with a couch and a television']}

## Visualiser les resultats

Les planches sont dans `reports/algorithm_tests/latest/image_to_image/`.

A verifier visuellement:

- `image_embedding_only`: voisins purement visuels,
- `image_caption_hybrid`: voisins visuels avec contexte caption,
- `01_filter_by_image`: filtre personne par InsightFace pour les cas avec visage.

## Lecture produit

Le benchmark montre que l'hybride image+caption ameliore la coherence caption par rapport au visuel seul, mais il ne remplace pas le filtre par personne. Pour le produit, l'interface doit proposer les deux comportements: recherche visuelle globale et filtre personne precis.